# Estudio 06 · Fase 1b · Consistencia del clima: lo descargado frente a ERA5 fijo (2017-2025)

`openmeteo.py` descargó 2017-2025 sin fijar `models`, así que Open-Meteo usó su «Best match», que según su
documentación combina IFS HRES, ERA5 y ERA5-Land (https://open-meteo.com/en/docs/historical-weather-api). El
modelo elegido puede no ser el mismo en todos los años. Antes de usar el clima 2000-2016, este notebook compara,
semana por semana y en los mismos distritos, lo ya descargado con ERA5 fijado explícitamente.

- **Entrada:** `data/bronze/meteo/cache_meteo/` (actual) y `data/bronze/meteo/cache_meteo_era5_2017_2025/`
  (celda A del notebook 02 del pipeline, la corre el equipo). Solo lee; no descarga ni escribe en `data/`.
- **Agregación:** la misma del notebook 03 del pipeline (`aggregate_to_weekly`).
- **Salida:** `docs/metricas/fase1b_consistencia.json` y `docs/figuras/fase1b/`.
- **Qué se mira:** la diferencia ERA5 − actual por variable (media, MAE, máximo, correlación, semanas iguales) y
  **por año**. Un salto entre años indica que el «Best match» cambió de modelo dentro de 2017-2025.
- La decisión (descargar 2000-2016 con «Best match», con ERA5 fijo o 2000-2025 con un solo modelo, que movería
  gold y la paridad de la V2) la toma Rosa. Este notebook no la toma.


In [ ]:
import json
import sys

import matplotlib.pyplot as plt
import pandas as pd

from denguekay.config.paths import CACHE_METEO, CACHE_METEO_ERA5_2017_2025, RESEARCH

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase1b_climate as cl

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
METRICS = STUDY / "docs" / "metricas" / "fase1b_consistencia.json"
FIGURES = STUDY / "docs" / "figuras" / "fase1b"
if not any(CACHE_METEO_ERA5_2017_2025.glob("distrito_*.parquet")):
    raise RuntimeError("Falta el caché ERA5: corran la celda A del notebook 02 del pipeline.")


## 1 · Cobertura: las dos descargas deben tener las mismas semanas


In [ ]:
res = cl.run_comparison(CACHE_METEO_ERA5_2017_2025, CACHE_METEO)
res["cobertura"]


## 2 · Diferencia ERA5 − actual por variable, 2017-2025


In [ ]:
res["por_variable"].round(4)


## 3 · Por año: ¿cambia la diferencia entre años?


In [ ]:
by_year = res["por_variable_anio"]
for stat in ("diferencia_media", "mae", "correlacion", "iguales"):
    print(stat)
    display(by_year.pivot(index="variable", columns="anio", values=stat).round(3))


In [ ]:
FIGURES.mkdir(parents=True, exist_ok=True)
variables = cl.WEEKLY_VARIABLES
fig, axes = plt.subplots(3, 3, figsize=(13, 9), sharex=True)
for ax, v in zip(axes.flat, variables):
    t = by_year[by_year.variable.eq(v)]
    ax.plot(t.anio, t.diferencia_media, marker="o", label="diferencia media")
    ax.plot(t.anio, t.mae, marker="s", label="MAE")
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_title(v)
axes.flat[0].legend(fontsize=8)
fig.suptitle("Clima semanal: ERA5 fijo − descargado (Best match), por año epidemiológico, 65 distritos\n"
             "Unidades de cada variable: °C, mm, %, km/h, MJ/m²")
fig.tight_layout()
fig.savefig(FIGURES / "diferencia_era5_por_anio.png", dpi=130)


## 4 · Cifras guardadas


In [ ]:
content = {
    "cobertura": res["cobertura"],
    "modelo_fijo_cache": CACHE_METEO_ERA5_2017_2025.name,
    "por_variable": res["por_variable"].to_dict(orient="records"),
    "por_variable_anio": by_year.to_dict(orient="records"),
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
METRICS.write_text(json.dumps(content, ensure_ascii=False, indent=1), encoding="utf-8")
METRICS
